In [ ]:
CONFIG = {
    "experiment": "notebook_paths",
    "display_simulation": True,
    "visualization": {
        "playback_speed": 1.0,
        "max_fps": 60,
        "ground_height_m": -1.0,
    },
    "ik": {
        # Use "nimble" for differentiable IK or "opensim" for legacy IK.
        "backend": "nimble",
        
        # Brace wrist motion and forearm pronation/supination.
        "lock_wrist": True,
        "locked_wrist_degrees": {
            # KINARM grip: palm/opening faces upward.
            "pro_sup": -30.0,
            "deviation": 0.0,
            "flexion": 0.0,
        },
        "coordinate_regularization_weight": 0.001,
        "nimble": {
            "wsl_distribution": "Ubuntu",
            "python": "/home/braydenk/.venvs/motor-meta-nimble/bin/python",
            "iterations": 12,
            "damping": 1e-5,
            "posture_weight": 1e-5,
        },
    },
    
    "proprioception": {
        # May be absolute or relative to the repository. Set to None to derive
        # the checkpoint directory from model_family and the two seeds.
        "model_path": "trained_models/experiment_causal_flag-pcr_optimized_linear_extended_5_5_letter_reconstruction_joints/spatiotemporal_4_8-8-32-64_7171_0_9",
        "model_family": "extended",
        "coefficient_seed": 0,
        "training_seed": 9,
    },
    
    "path": {
        "generator": "process/generate_paths/draw.py", #or generatereachpath.py 
        "sample_rate_hz": 240,
        "reach_cm": 10.0,
        "max_displacement_cm": 45.0,
        "workspace_ellipse": {
            "center_cm": [-10.0, 0.0],
            "radii_cm": [30.0, 40.0],
        },
        "path_speed_cm_s": 20.0,
        "return_to_rest": False,
        "rest_pose_degrees": {
            "elv_angle": 20.0,
            "shoulder_elv": 40.0,
            "shoulder_rot": 25.0,
            "elbow_flexion": 85.0,
        },
        "timing_seconds": {
            "hold_before": 1.65,
            "reach": 0.50,
            "hold_target": 0.50,
            "return": 0.50,
            "hold_after": 1.65,
        },
    },
}

In [14]:
# Materialize CONFIG as YAML and prepare the shared runner.
# The generated file is under ignored outputs/.
from pathlib import Path
import yaml
from run_pipeline import Pipeline, STAGE_NAMES

CONFIG_PATH = Path("outputs/notebook_config.yaml").resolve()
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
with CONFIG_PATH.open("w", encoding="utf-8") as stream:
    yaml.safe_dump(CONFIG, stream, sort_keys=False)

pipeline = Pipeline(CONFIG_PATH)
print(f"Configuration: {CONFIG_PATH}")
print(f"Outputs:       {pipeline.output_dir}")
print(f"Stages:        {', '.join(STAGE_NAMES)}")

Configuration: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\outputs\notebook_config.yaml
Outputs:       C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\outputs\notebook_paths
Stages:        path, inverse-kinematics, motion, muscle-lengths, muscle-signals, inference


## 1. Generate paths
This opens the drawing window when the draw generator is selected above.

In [15]:
pipeline.run_stage("path")


=== path: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\process\generate_paths\draw.py ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\generate_paths\\draw.py'], returncode=0)

## 2. Inverse kinematics

In [16]:
pipeline.run_stage("inverse-kinematics")


=== inverse-kinematics: Nimble Physics (WSL) ===


CompletedProcess(args=['wsl', '-d', 'Ubuntu', '--', 'env', 'MOTOR_META_EXPERIMENT=notebook_paths', 'MOTOR_META_CONFIG=/mnt/c/Users/brayk/OneDrive/Documents/CMU-Classes/PhiLab/MotorMetamersuPNC/outputs/notebook_config.yaml', '/home/braydenk/.venvs/motor-meta-nimble/bin/python', '/mnt/c/Users/brayk/OneDrive/Documents/CMU-Classes/PhiLab/MotorMetamersuPNC/process/nimble_ik.py'], returncode=0)

## 3. Generate OpenSim motion

In [17]:
pipeline.run_stage("motion")


=== motion: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\process\gencenterout.py ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\gencenterout.py'], returncode=0)

## Review motion
Optional. This launches the same OpenSim playback/control UI used by the command-line pipeline.

In [18]:
pipeline.display_simulation()


=== motion-review: OpenSim visualizer ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\utils\\display_sim.py'], returncode=0)

## 4. Extract muscle lengths

In [7]:
pipeline.run_stage("muscle-lengths")


=== muscle-lengths: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\process\extractcenterout.py ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\extractcenterout.py'], returncode=0)

## 5. Compute spindle signals

In [8]:
pipeline.run_stage("muscle-signals")


=== muscle-signals: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\process\computefrcenterout.py ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\computefrcenterout.py'], returncode=0)

## 6. Run neural inference
This requires the pretrained checkpoint described in the README.

In [ ]:
pipeline.run_stage("inference")

## Differentiability status
Some individual components, including the Nimble IK operation and neural network, are differentiable. This stage-oriented notebook currently saves and reloads artifacts between processes, so it does **not** preserve an end-to-end computation graph and cannot yet backpropagate a loss to the input path. Building the in-memory differentiable runner and exposing it here is tracked in `TODO-differentiable-pipeline.md`.